In [ ]:
import numpy as np
from sklearn.metrics import make_scorer

In [ ]:
# NumPy fundamental vectorization tricks

a = np.array([1, 2, 4])
b = np.array([2, 3, 3])
eps = 1e-13

# Conditions and masks
np.where(a > 1, a * 2, 404)
exprsn = (a > 1) * a * 2 + (a <= 1) * 404 # math equal to np.where (mask product)
np.maximum(a, b)
np.maximum(a, 0) # all negative numbers became zeros

# Defended math
np.log1p(a) # calculate log(1 + x) ~ x - x^2/2 + x^3/3, np.log(1 + x) works bad with x ~ 0
np.clip(a, eps, None) # we take ~ 0, but not 0

# Broadcasting
brdcstn = a - np.mean(a)

# Vectorizationed checking
np.any(a < 0)
np.all(b > 2)

# Aggregation
np.mean(a)
np.median(a)
np.max(a)
np.var(a) # !!! very useful

np.float64(1.5555555555555554)

**Принятые обозначения**

- $y$ — вектор истинных значений (иногда обозначаются вектора полужирным через `mathbf{...}`: $\mathbf{y}$)
- $\hat{y}$ — вектор предсказаний модели
- $y_i$ — $i$-й элемент истинных данных
- $\hat{y}$ — $i$-й элемент предсказания модели
- $\bar{y}$ — среднее арифметическое истинных данных
- $n$ или $N$ — количество объектов в выборке
- $e_i$ — ошибка для $i$-го объекта

- $\text{Var}(y)$ или $\sigma^2$ — дисперсия
- $\mathbb{E}[y]$ или $\mu$ — математическое ожидание
- $L(\theta)$ или $\mathcal{L}[\theta]$ — функция правдоподобия
- $\ell(\theta)$ — логарифм правдоподобия
- $D(y, \hat{y})$ — девианс общий (для выборки)
- $d(y_i, \hat{y_i})$ — дквианс единичный (для точки)
- $\alpha$ или $q$ — квантиль (Pinball Loss)
- $p$ — индекс распределения Твинди (Power)
- $\phi$ — параметр масштаба (дисперсии) в Твиди
- $R^2$ — коэффициент детерминации
- $D^2$ — доля объяснённого девианса

In [ ]:
# How to build custom metric in scikit-learn
def custom_mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

# Wrap the function. Set greater_is_better to False because it's an error metric.
my_mse_scorer = make_scorer(custom_mse, greater_is_better=False)

# Supervised learning

## Regression

**Можно выделить 5 типов метрик регресси в `scikit-learn`**

1. **Базовые**:
  - `max_error`
  - `mean_absolute_error`
  - `mean_squared_error`
  - `median_absolute_error`
  - `root_mean_squared_error`
2. **Относительные**:
  - `mean_absolute_percentage_error`
  - `mean_squared_log_error`
  - `r2_score`
  - `root_mean_squared_log_error`
3. **Доля объяснённого**:
  - `d2_absolute_error_score`
  - `d2_pinball_score`
  - `d2_tweedie_score`
  - `explainde_variance_score`
4. **Квантильные**:
  - `mean_pinball_loss`
5. **Статистические**:
  - `mean_gamma_deviance`
  - `mean_poisson_deviance`
  - `mean_tweedie_deviance`

### 1. Базовые метрики: `MSE, RMSE, MAE, MedAE, MaxError`.

$$
MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y_i})^2
$$

In [ ]:
def MSE(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

$$
RMSE = \sqrt{MSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y_i})^2}
$$

In [ ]:
def RMSE(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

$$
MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y_i}|
$$

In [ ]:
def MAE(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

$$
MedAE = \text{median}(|y_1 - \hat{y_1}|, ..., |y_n - \hat{y_n}|)
$$

In [ ]:
def MedAE(y_true, y_pred):
    return np.median(np.abs(y_true - y_pred))

$$
MaxError = \max_{i=\overline{1,n}}(|y_i - \hat{y_i}|)
$$

In [ ]:
def MaxError(y_true, y_pred):
    return np.max(np.abs(y_true - y_pred))

### 2. Отсносительные метрики: `MAPE, R², MSLE, RMSLE`.

$$
MAPE = \frac{1}{n} \sum_{i=1}^{n} \left|\frac{y_i - \hat{y_i}}{y_i}\right|
$$

In [ ]:
def MAPE(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true))

$$
R^2 = 1 - \frac{MSE(y, \hat{y})}{MSE(y, \bar{y})} = 1 - \frac{ \sum_{i=1}^{n} (y_i - \hat{y_i})^2 }{ \sum_{i=1}^{n} (y_i - \bar{y_i})^2 }
$$

In [ ]:
def R2(y_true, y_pred):
    return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2)

$$
MSLE = \frac{1}{n} \sum_{i=1}^{n} (\text{log}(y_i + 1) - \text{log}(\hat{y_i} + 1))^2
$$

In [ ]:
def MSLE(y_true, y_pred):
    return np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2)

$$
RMSLE = \sqrt{MSLE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (\text{log}(y_i + 1) - \text{log}(\hat{y_i} + 1))^2}
$$

In [ ]:
def RMSLE(y_true, y_pred):
    return np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))

### 3. Статистические метрики (Deviance-Based): `Poisson, Gamma, Tweedie`.

$$
MPD = \frac{1}{n} \sum_{i=1}^{n} 2 (y_i \cdot \text{log}\frac{y_i}{\hat{y_i}} - (y_i - \hat{y_i}))
$$

In [ ]:
def MPD(y_true, y_pred):
    return np.mean(2 * (y_true * np.log(y_true / y_pred) - (y_true - y_pred)))

$$
MGD = \frac{1}{n} \sum_{i=1}^{n} 2 \left(\frac{y_i}{\hat{y_i}} - \text{log}\frac{y_i}{\hat{y_i}} - 1 \right)
$$

In [ ]:
def MGD(y_true, y_pred):
    return np.mean(2 * (y_true / y_pred -  np.log(y_true / y_pred) - 1))

$$
MTD = \frac{1}{n} \sum_{i=1}^{n} 2 \left(\frac{y_i^{2-p}}{(1-p)(2-p)} - \frac{y_i\hat{y_i}^{1-p}}{1-p} - \frac{\hat{y_i}^{2-p}}{2-p}\right),
\quad p \notin \{1, 2\}
$$

In [ ]:
def MTD(y_true, y_pred, p):
    if np.isclose(p, 1) or np.isclose(p, 2):
        raise ValueError(
            f"Параметр {p} для формулы не допустим."
    )
    y_2p = y_true ** (2 - p) / ((1 - p) * (2 - p))
    yyh_1p = y_true * y_pred ** (1 - p) / (1 - p)
    yh_2p = y_pred ** (2 - p) / (2 - p)
    return np.mean(2 * (y_2p - yyh_1p - yh_2p))

### 4. Квантильные метрики: `PinballLoss`.

$$
PinballLoss =
\begin{cases}
{
\alpha (y - \hat{y}),\quad y \ge \hat{y} \\\
(1 - \alpha) (y - \hat{y}),\quad y < \hat{y}
}
\end{cases}
$$

In [ ]:
def PL(y_true, y_pred, alpha, reduction=True):
    loss = np.maximum(alpha * (y_true - y_pred), (alpha - 1) * (y_true - y_pred))
    return np.mean(loss) if reduction else loss

### 5. Доля объяснённого: `EVS, D2AES, D2PS, D2TS`.








$$
EVS = \frac{1}{n} \sum_{i=1}^n \frac{\text{Var}(y_i - \hat{y_i})}{\text{Var}(y_i)}
$$

In [ ]:
def ELS(y_true, y_pred):
    var_sub = np.var(y_true - y_pred)
    var_tru = np.var(y_true)
    if var_true == 0:
        return 1.0 if var_sub == 0 else 0.0
    return np.mean(var_sub / var_tru)

$$
D2AES = 1 - \frac{\sum_{i=1}^{n} |y_i - \hat{y_i}|}{\sum_{i=1}^{n} | y_i - \text{median}(y)|}
$$

In [ ]:
def D2AES(y_true, y_pred):
    return 1 - np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true - np.median(y_true)))

$$
D2PS = 1 - \frac{\sum_{i=1}^n \max(\alpha(y_i - \hat{y_i}), (\alpha - 1)(y_i - \hat{y_i}))}{\sum_{i=1}^n \max(\alpha(y_i - \bar{y_i}), (\alpha - 1)(y_i - \bar{y_i}))}
$$

In [ ]:
def D2PS(y_true, y_pred, alpha):
    numerator = np.maximum(alpha * (y_true - y_pred), (alpha - 1) * (y_true - y_pred))
    denumerator = np.maximum(alpha * (y_true - np.mean(y_true)), (alpha - 1) * (y_true - np.mean(y_true)))
    return 1 - np.sum(numerator) / np.sum(denumerator)


$$
D2TS = 1 - \frac{\sum_{i=1}^{n} \left(\frac{y_i^{2-p}}{(1-p)(2-p)} - \frac{y_i\hat{y_i}^{1-p}}{1-p} - \frac{\hat{y_i}^{2-p}}{2-p}\right)}{ \sum_{i=1}^{n} \left(\frac{y_i^{2-p}}{(1-p)(2-p)} - \frac{y_i\bar{y_i}^{1-p}}{1-p} - \frac{\bar{y_i}^{2-p}}{2-p}\right)}
$$

In [ ]:
def D2TS(y_true, y_pred, p):
    if np.isclose(p, 1) or np.isclose(p, 2):
        raise ValueError(
            f"Параметр {p} для формулы не допустим."
    )
    y_2p = y_true ** (2 - p) / ((1 - p) * (2 - p))
    yyh_1p = y_true * y_pred ** (1 - p) / (1 - p)
    yh_2p = y_pred ** (2 - p) / (2 - p)
    numerator = np.sum(y_2p - yyh_1p - yh_2p)
    yyb_1p = y_true * np.mean(y_true) ** (1 - p) / (1 - p)
    yb_2p = np.mean(y_true) ** (2 - p) / (2 - p)
    denumerator = np.sum(y_2p - yyb_1p - yb_2p)
    return 1 - numerator / denumerator

## Classification

# Unsupervised learning

## Clasterization